"""
# ProlabDep Tutorial

This notebook demonstrates how to use the ProlabDep package to process and analyze wastewater treatment plant data.
"""

## 1. Installation and Setup

First, let's install the package if you haven't already:

```bash
pip install prolabdep
```

Now, let's import the necessary modules:

In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

# Import ProlabDep modules
from prolabdep.api import Client
from prolabdep.visualization import Visualizer

# Configure matplotlib for better display in notebook
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['figure.dpi'] = 100

# Enable inline plotting
%matplotlib inline

## 2. Initialize Client

The `Client` class is the main entry point for interacting with the ProlabDep package:

In [3]:
# Initialize client with a local database
client = Client(db_path="tutorial_data.db")

print(f"Client initialized with database at {client._data_cache}")

2025-05-23 12:53:01,264 - prolabdep.core.database - INFO - Database initialized at E:\python_apps\prolabwwtp\prolabdep\examples\tutorial_data.db


Client initialized with database at {}


## 3. Load Sample Data

For this tutorial, we'll either use your own data file or create some synthetic data:

In [4]:
# Define file path to your data
data_file = "TUTTO_2024_25.csv"  # Replace with your data file path

# Check if the file exists
if os.path.exists(data_file):
    print(f"Found data file at {data_file}")
    # Import data from file
    data_id = client.import_data(data_file)
    print(f"Data imported with ID: {data_id}")
else:
    print(f"Data file not found at {data_file}, using synthetic data instead")
    
    # Create synthetic data for demonstration purposes
    # First, let's create sample metadata
    sample_ids = [f"S{i:04d}" for i in range(1, 101)]
    dates = [datetime(2024, 1, 1) + timedelta(days=i) for i in range(100)]
    sites = ["North WWTP", "South WWTP"] * 50
    sampling_points = ["Inlet", "Outlet"] * 50
    
    # Create synthetic parameters
    parameters = {
        "COD": {"unit": "mg/L", "values": np.random.normal(250, 50, 100)},
        "BOD": {"unit": "mg/L", "values": np.random.normal(120, 30, 100)},
        "TSS": {"unit": "mg/L", "values": np.random.normal(180, 40, 100)},
        "pH": {"unit": "pH", "values": np.random.normal(7.2, 0.5, 100)},
        "FLOW": {"unit": "m3/h", "values": np.random.normal(500, 100, 100)}
    }
    
    # Create sample dataframes for each parameter
    dfs = []
    for param, info in parameters.items():
        df = pd.DataFrame({
            "sample_id": sample_ids,
            "date": dates,
            "site": sites,
            "sampling_point": sampling_points,
            "parameter_name": param,
            "unit": info["unit"],
            "value": info["values"]
        })
        dfs.append(df)
    
    # Combine all parameter dataframes
    synthetic_data = pd.concat(dfs, ignore_index=True)
    print(f"Created synthetic data with {len(synthetic_data)} measurements")

2025-05-23 12:54:31,110 - prolabdep.processors.csv_processor - INFO - Processing CSV file: TUTTO_2024_25.csv


Found data file at TUTTO_2024_25.csv


E:\python_apps\prolabwwtp\prolabdep\prolabdep\processors\csv_processor.py:69: DtypeWarning: Columns (7,8,9,10,11,12,13,14,15,48,51,54,55,56,57,60,61,62,104,105,106,107,117,222,223,224,225,226,227,228,229,230,231,232,236,242,243,244,245,246,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435) have mixed types. S

Data imported with ID: 577de6b7-d4c8-488a-852c-1ac8b09e99da


## 4. Explore Parameters

Let's look at what parameters are available in our dataset:

In [7]:
# Get parameters from database
parameters = client.get_parameters()

if not parameters.empty:
    print(f"Found {len(parameters)} parameters:")
    display(parameters.head())
else:
    # If using synthetic data, show parameters
    print("Parameters in synthetic data:")
    print(list(parameters.keys()))

Found 417 parameters:


,code,name,method,unit,description
0,TEMPERATURAACC@APATIRSAVOL.1N°2100@°C,TEMPERATURAACC,APATIRSAVOL.1N°2100,°C,Temperatura in accettazione (°C)
1,111TRICLOROETANO_DEP_OA@IRSAN°5150@mG/L,111TRICLOROETANO_DEP_OA,IRSAN°5150,mG/L,"1,1,1 -Tricloroetano (mg/l)"
2,CARBONIOTETRACLORURO_DEP_OA@IRSAN°5150@mG/L,CARBONIOTETRACLORURO_DEP_OA,IRSAN°5150,mG/L,Carbonio tetracloruro (mg/l)
3,BROMODICLOROMETANO_DEP_OA@IRSAN°5150@mG/L,BROMODICLOROMETANO_DEP_OA,IRSAN°5150,mG/L,Bromodiclorormetano (mg/l)
4,BROMOFORMIO_DEP_OA@IRSAN°5150@mG/L,BROMOFORMIO_DEP_OA,IRSAN°5150,mG/L,Bromoformio (mg/l)


## 5. Retrieve Time Series Data

Now, let's retrieve time series data for a specific parameter (e.g., COD):

In [6]:
# Get time series data for COD parameter
parameter = "COD"
site = "Dep. S.Colombano"  # Update this to match your data
sampling_point = "uscita"  # Update this to match your data

timeseries = client.get_parameter_timeseries(

    parameter=parameter,
    site=site,
    sampling_point=sampling_point
)

if not timeseries.empty:
    print(f"Retrieved {len(timeseries)} {parameter} measurements")
    display(timeseries.head())
else:
    print(f"No {parameter} data found for {site} {sampling_point}")
    
    # If using synthetic data, filter it
    if 'synthetic_data' in locals():
        timeseries = synthetic_data[
            (synthetic_data['parameter_name'] == parameter) & 
            (synthetic_data['site'] == site) & 
            (synthetic_data['sampling_point'] == sampling_point)
        ]
        print(f"Using synthetic data: {len(timeseries)} {parameter} measurements")
        display(timeseries.head())

TimeoutError: QueuePool limit of size 5 overflow 10 reached, connection timed out, timeout 30.00 (Background on this error at: https://sqlalche.me/e/20/3o7r)

## 6. Basic Statistics and Analysis

Let's calculate some basic statistics for the time series data:

In [ ]:
# Calculate statistics
stats = client.analyze_statistics(timeseries)

print(f"{parameter} Statistics:")
for stat, value in stats.items():
    print(f"  {stat}: {value}")

## 7. Time Series Visualization

Now, let's create a time series plot:

In [ ]:
# Create time series plot
fig = client.plot_time_series(
    timeseries,
    parameter_name=parameter,
    title=f"{parameter} Time Series for {site} {sampling_point}",
    include_trend=True
)

plt.tight_layout()
plt.show()

## 8. Data Resampling

Let's resample the data to a different frequency (e.g., weekly averages):

In [ ]:
# Resample to weekly data
weekly_data = client.resample_timeseries(
    timeseries,
    frequency='W',  # Weekly
    method='mean'   # Average
)

print(f"Resampled to {len(weekly_data)} weekly data points")
display(weekly_data.head())

# Create time series plot of resampled data
fig = client.plot_time_series(
    weekly_data,
    parameter_name=f"Weekly {parameter}",
    title=f"Weekly Average {parameter} for {site} {sampling_point}",
    include_trend=True
)

plt.tight_layout()
plt.show()

## 9. Compare Different Sites or Sampling Points

Let's compare data from different sites or sampling points:

In [ ]:
# Get data for another site or sampling point
alt_sampling_point = "Inlet"  # Change this based on your data

alt_timeseries = client.get_parameter_timeseries(
    parameter=parameter,
    site=site,
    sampling_point=alt_sampling_point
)

if alt_timeseries.empty and 'synthetic_data' in locals():
    # Use synthetic data if needed
    alt_timeseries = synthetic_data[
        (synthetic_data['parameter_name'] == parameter) & 
        (synthetic_data['site'] == site) & 
        (synthetic_data['sampling_point'] == alt_sampling_point)
    ]

# Create comparison plot using the Visualizer class directly
visualizer = Visualizer()

if not timeseries.empty and not alt_timeseries.empty:
    fig = visualizer.plot_comparison(
        [timeseries, alt_timeseries],
        labels=[f"{sampling_point}", f"{alt_sampling_point}"],
        title=f"{parameter} Comparison at {site}"
    )
    
    plt.tight_layout()
    plt.show()

## 10. Calculate Mass Flow

If we have concentration and flow data, we can calculate mass flow:

In [ ]:
# Get flow data
flow_parameter = "FLOW"
flow_data = client.get_parameter_timeseries(
    parameter=flow_parameter,
    site=site,
    sampling_point=sampling_point
)

if flow_data.empty and 'synthetic_data' in locals():
    # Use synthetic data if needed
    flow_data = synthetic_data[
        (synthetic_data['parameter_name'] == flow_parameter) & 
        (synthetic_data['site'] == site) & 
        (synthetic_data['sampling_point'] == sampling_point)
    ]

# Calculate mass flow if we have both concentration and flow data
if not timeseries.empty and not flow_data.empty:
    mass_flow = client.calculate_mass_flow(
        concentration=timeseries,
        flow_parameter=flow_parameter,
        flow_data=flow_data,
        output_unit='kg/d'  # kilograms per day
    )
    
    print(f"Calculated {len(mass_flow)} mass flow data points")
    display(mass_flow.head())
    
    # Plot mass flow
    fig = client.plot_time_series(
        mass_flow,
        parameter_name=f"{parameter} Load",
        title=f"{parameter} Mass Flow at {site} {sampling_point}",
        include_trend=True
    )
    
    plt.tight_layout()
    plt.show()

## 11. Export Data

Finally, let's export our data to different formats:

In [ ]:
# Export to Excel
excel_file = f"{parameter}_data.xlsx"
client.export_to_excel(timeseries, excel_file)
print(f"Data exported to Excel: {excel_file}")

# Export to CSV
csv_file = f"{parameter}_data.csv"
client.export_to_csv(timeseries, csv_file)
print(f"Data exported to CSV: {csv_file}")

## Conclusion

In this tutorial, we've learned how to:

1. Import data using ProlabDep
2. Retrieve and analyze time series data
3. Calculate statistics and visualize trends
4. Resample data to different frequencies
5. Compare data from different sources
6. Calculate mass flow
7. Export data to different formats

The ProlabDep package provides a comprehensive set of tools for processing and analyzing wastewater treatment plant data from LIMS exports. 